### Temporal Feature Engineering

Transform historical sensor measurements into leakage-safe temporal features
that can help machine-learning models predict Remaining Useful Life (RUL).

##### Key Questions

1. Which temporal information can be extracted from sensor history?
2. Can recent sensor changes help predict RUL?
3. Can rolling statistics reduce short-term sensor noise?
4. Can temporal trends capture degradation?
5. Can these features be constructed without future information?
6. Can temporal operations remain correctly separated by engine?
7. Which feature families are useful enough to carry forward?


#### Load the Phase 1 Dataset

The FD001 training, test, and RUL datasets were loaded using the reusable
`load_dataset()` function from `src/data_loader.py`.

The training dataset contains 20,631 observations from 100 engines, while
the test dataset contains 13,096 observations. The provided test RUL file
contains one RUL value for each of the 100 test engines.

Temporal Feature Engineering starts from the same underlying sensor observations established
during Data investigation. The purpose of this is not to repeat the sensor
investigation, but to transform historical sensor information into
predictive temporal features

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.data_loader import load_dataset

train_df, test_df, rul_df = load_dataset("FD001")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("RUL shape:", rul_df.shape)

Train shape: (20631, 26)
Test shape: (13096, 26)
RUL shape: (100, 1)


#### Reconstruct the Rul Engineering RUL Target

Because the reusable data loader returns the original C-MAPSS datasets,
the final RUL target defined during Phase 1 was reconstructed for use in
this notebook.

The raw training RUL is calculated separately for each engine as:

`RUL_raw = maximum_cycle_of_engine - current_cycle`

The final target uses the Phase 1 cap of 125 cycles:

`RUL = min(RUL_raw, 125)`

This ensures that Phase 2 uses exactly the same target definition established
in Phase 1 rather than introducing a new target formulation.

In [2]:
RUL_CAP = 125

train_df["max_cycle"] = (
    train_df.groupby("unit")["cycle"]
    .transform("max")
)

train_df["RUL_raw"] = (
    train_df["max_cycle"] - train_df["cycle"]
)

train_df["RUL"] = (
    train_df["RUL_raw"].clip(upper=RUL_CAP)
)

In [3]:
train_df[["unit", "cycle", "RUL_raw", "RUL"]].head()

,unit,cycle,RUL_raw,RUL
0,1,1,191,125
1,1,2,190,125
2,1,3,189,125
3,1,4,188,125
4,1,5,187,125


#### Validate the Target

The reconstructed target was checked for its minimum, maximum, and missing
values.

Results:

- Minimum RUL: 0
- Maximum RUL: 125
- Missing RUL values: 0

Therefore, the Phase 1 target has been successfully reconstructed and is
ready to be used with the temporal features developed in this phase.

In [4]:
print("RUL minimum:", train_df["RUL"].min())
print("RUL maximum:", train_df["RUL"].max())
print("RUL missing values:", train_df["RUL"].isna().sum())

RUL minimum: 0
RUL maximum: 125
RUL missing values: 0


#### Identify the Temporal Structure

The training data contains:

- 100 unique engines
- 20,631 total observations
- Maximum observed cycle: 362

Each engine contains a chronological sequence of observations:

Engine → Cycle 1 → Cycle 2 → Cycle 3 → ...

Temporal features must therefore be calculated within each engine rather
than across the complete dataset.

In [5]:
print("Number of engines:", train_df["unit"].nunique())

print(
    "Total observations:",
    len(train_df)
)

print(
    "Maximum cycle:",
    train_df["cycle"].max()
)

Number of engines: 100
Total observations: 20631
Maximum cycle: 362


In [6]:
train_df[
    ["unit", "cycle"]
].head(10)

,unit,cycle
0,1,1
1,1,2
2,1,3
3,1,4
4,1,5
5,1,6
6,1,7
7,1,8
8,1,9
9,1,10


#### Verify Chronological Ordering

The cycle ordering was checked separately for every engine.

Result:

`All engines chronologically sorted: True`

Therefore, the observations are already ordered chronologically within each
engine.

This is important because lag and rolling operations depend on the order
of observations.

In [7]:
is_sorted = (
    train_df
    .groupby("unit")["cycle"]
    .apply(lambda x: x.is_monotonic_increasing)
)

print("All engines chronologically sorted:", is_sorted.all())

All engines chronologically sorted: True


#### Explicitly Sort the Dataset

Even though the data was already chronologically ordered, the dataframe was
explicitly sorted by:

`unit → cycle`

This establishes a clear ordering assumption before temporal operations
are performed.

In [8]:
train_df = train_df.sort_values(
    ["unit", "cycle"]
).reset_index(drop=True)

In [9]:
train_df[["unit", "cycle"]].head(10)

,unit,cycle
0,1,1
1,1,2
2,1,3
3,1,4
4,1,5
5,1,6
6,1,7
7,1,8
8,1,9
9,1,10


#### Define the Sensor Feature Set

The FD001 dataset contains 21 sensor measurements:

`sensor_1` through `sensor_21`

These sensors form the primary raw feature family from which temporal
features will be constructed.

The purpose of Phase 2 is not to rediscover sensor behavior, which was
already investigated in Phase 0, but to transform historical sensor
measurements into features representing recent change, variability, and
trend.

---

In [10]:
sensor_cols = [
    f"sensor_{i}"
    for i in range(1, 22)
]

print("Number of sensors:", len(sensor_cols))
print(sensor_cols)

Number of sensors: 21
['sensor_1', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_5', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21']


In [11]:
raw_features = train_df[
    ["unit", "cycle"] + sensor_cols
].copy()

#### Establish the Raw Feature Representation

A raw-sensor feature representation was created using:

- unit
- cycle
- sensor_1 through sensor_21

This produced a feature representation with 20,631 observations and
23 columns.

This representation will serve as the reference point for later
comparisons between raw sensor information and engineered temporal
features.

---

In [12]:
raw_features.head()

,unit,cycle,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,sensor_6,sensor_7,sensor_8,...,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
0,1,1,518.67,641.82,1589.70,1400.60,14.62,21.61,554.36,2388.06,...,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,518.67,642.15,1591.82,1403.14,14.62,21.61,553.75,2388.04,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,518.67,642.35,1587.99,1404.20,14.62,21.61,554.26,2388.08,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,518.67,642.35,1582.79,1401.87,14.62,21.61,554.45,2388.11,...,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,518.67,642.37,1582.85,1406.22,14.62,21.61,554.00,2388.06,...,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


In [13]:
print("Raw feature shape:", raw_features.shape)

Raw feature shape: (20631, 23)



#### First Temporal Feature — One-Cycle Lag

The first temporal feature was created using the previous cycle's value of
`sensor_2`.

The feature is calculated separately for each engine using a grouped
one-step shift.

Conceptually:

Current cycle → current sensor value

Previous cycle → lag-1 sensor value

For example:

Cycle 2:
`sensor_2_lag1 = sensor_2 at Cycle 1`

Cycle 3:
`sensor_2_lag1 = sensor_2 at Cycle 2`

The first cycle of every engine has no previous observation, so its lag value
is expected to be `NaN`.

In [14]:
train_df["sensor_2_lag1"] = (
    train_df.groupby("unit")["sensor_2"]
    .shift(1)
)

In [16]:
train_df[
    ["unit", "cycle", "sensor_2", "sensor_2_lag1"]
].head(10)

,unit,cycle,sensor_2,sensor_2_lag1
0,1,1,641.82,NaN
1,1,2,642.15,641.82
2,1,3,642.35,642.15
3,1,4,642.35,642.35
4,1,5,642.37,642.35
5,1,6,642.10,642.37
6,1,7,642.48,642.10
7,1,8,642.56,642.48
8,1,9,642.12,642.56
9,1,10,641.71,642.12


In [17]:
first_cycles = (
    train_df
    .sort_values(["unit", "cycle"])
    .groupby("unit")
    .head(1)
)

first_cycles[
    ["unit", "cycle", "sensor_2", "sensor_2_lag1"]
].head(10)

,unit,cycle,sensor_2,sensor_2_lag1
0,1,1,641.82,NaN
192,2,1,641.89,NaN
479,3,1,642.04,NaN
658,4,1,642.60,NaN
847,5,1,641.77,NaN
1116,6,1,642.73,NaN
1304,7,1,642.38,NaN
1563,8,1,643.18,NaN
1713,9,1,642.18,NaN
1914,10,1,641.92,NaN


#### 10. Verify Engine Boundaries

The first-cycle observations of every engine were inspected to verify that
lag calculations do not cross engine boundaries.

Result:

`First-cycle lag values that are not NaN: 0`

This confirms that the lag operation is correctly grouped by engine.

Therefore, the final cycle of one engine is not incorrectly used as the
previous observation for the first cycle of another engine.

This establishes an important temporal-feature engineering rule:

> All temporal operations must respect engine boundaries.

In [18]:
print(
    "First-cycle lag values that are not NaN:",
    first_cycles["sensor_2_lag1"].notna().sum()
)

First-cycle lag values that are not NaN: 0


In [19]:
train_df["sensor_2_diff1"] = (
    train_df.groupby("unit")["sensor_2"]
    .diff(1)
)

train_df[
    ["unit", "cycle", "sensor_2", "sensor_2_lag1", "sensor_2_diff1"]
].head(10)

,unit,cycle,sensor_2,sensor_2_lag1,sensor_2_diff1
0,1,1,641.82,NaN,NaN
1,1,2,642.15,641.82,0.33
2,1,3,642.35,642.15,0.20
3,1,4,642.35,642.35,0.00
4,1,5,642.37,642.35,0.02
5,1,6,642.10,642.37,-0.27
6,1,7,642.48,642.10,0.38
7,1,8,642.56,642.48,0.08
8,1,9,642.12,642.56,-0.44
9,1,10,641.71,642.12,-0.41


In [20]:
train_df[
    [
        "unit",
        "cycle",
        "sensor_2",
        "sensor_2_lag1",
        "sensor_2_diff1"
    ]
].head(20)

,unit,cycle,sensor_2,sensor_2_lag1,sensor_2_diff1
0,1,1,641.82,NaN,NaN
1,1,2,642.15,641.82,0.33
2,1,3,642.35,642.15,0.20
3,1,4,642.35,642.35,0.00
4,1,5,642.37,642.35,0.02
5,1,6,642.10,642.37,-0.27
6,1,7,642.48,642.10,0.38
7,1,8,642.56,642.48,0.08
8,1,9,642.12,642.56,-0.44
9,1,10,641.71,642.12,-0.41


In [21]:
print(
    "Missing sensor_2_diff1:",
    train_df["sensor_2_diff1"].isna().sum()
)

print(
    "Number of engines:",
    train_df["unit"].nunique()
)

Missing sensor_2_diff1: 100
Number of engines: 100


In [22]:
first_cycles = (
    train_df
    .sort_values(["unit", "cycle"])
    .groupby("unit")
    .head(1)
)

print(
    "First-cycle differences that are not NaN:",
    first_cycles["sensor_2_diff1"].notna().sum()
)

First-cycle differences that are not NaN: 0


In [23]:
for sensor in sensor_cols:
    train_df[f"{sensor}_diff1"] = (
        train_df.groupby("unit")[sensor]
        .diff(1)
    )

In [24]:
diff_cols = [f"{sensor}_diff1" for sensor in sensor_cols]

print("Number of difference features:", len(diff_cols))

print(diff_cols)

Number of difference features: 21
['sensor_1_diff1', 'sensor_2_diff1', 'sensor_3_diff1', 'sensor_4_diff1', 'sensor_5_diff1', 'sensor_6_diff1', 'sensor_7_diff1', 'sensor_8_diff1', 'sensor_9_diff1', 'sensor_10_diff1', 'sensor_11_diff1', 'sensor_12_diff1', 'sensor_13_diff1', 'sensor_14_diff1', 'sensor_15_diff1', 'sensor_16_diff1', 'sensor_17_diff1', 'sensor_18_diff1', 'sensor_19_diff1', 'sensor_20_diff1', 'sensor_21_diff1']


In [25]:
diff_missing = train_df[diff_cols].isna().sum()

print(diff_missing)

sensor_1_diff1     100
sensor_2_diff1     100
sensor_3_diff1     100
sensor_4_diff1     100
sensor_5_diff1     100
sensor_6_diff1     100
sensor_7_diff1     100
sensor_8_diff1     100
sensor_9_diff1     100
sensor_10_diff1    100
sensor_11_diff1    100
sensor_12_diff1    100
sensor_13_diff1    100
sensor_14_diff1    100
sensor_15_diff1    100
sensor_16_diff1    100
sensor_17_diff1    100
sensor_18_diff1    100
sensor_19_diff1    100
sensor_20_diff1    100
sensor_21_diff1    100
dtype: int64


In [26]:
print(
    "Total missing difference values:",
    train_df[diff_cols].isna().sum().sum()
)

Total missing difference values: 2100


In [27]:
train_df[diff_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
sensor_1_diff1,20531.0,0.000000,0.000000,0.0000,0.0000,0.0000,0.0000,0.0000
sensor_2_diff1,20531.0,0.006452,0.427006,-1.7500,-0.2800,0.0100,0.3000,1.5700
sensor_3_diff1,20531.0,0.078337,5.657544,-23.4800,-3.7000,0.0700,3.8900,23.1200
sensor_4_diff1,20531.0,0.138499,5.699366,-22.3500,-3.7100,0.1100,4.0100,25.0100
sensor_5_diff1,20531.0,0.000000,0.000000,0.0000,0.0000,0.0000,0.0000,0.0000
sensor_6_diff1,20531.0,0.000002,0.001887,-0.0100,0.0000,0.0000,0.0000,0.0100
sensor_7_diff1,20531.0,-0.012831,0.577331,-2.2300,-0.4000,-0.0200,0.3700,2.7000
sensor_8_diff1,20531.0,0.000923,0.042802,-0.1700,-0.0300,0.0000,0.0300,0.3500
sensor_9_diff1,20531.0,0.211973,5.903739,-22.1200,-3.7200,0.1800,4.2000,24.0000
sensor_10_diff1,20531.0,0.000000,0.000000,0.0000,0.0000,0.0000,0.0000,0.0000


In [28]:
train_df[diff_cols].mean().sort_values()

sensor_7_diff1    -0.012831
sensor_12_diff1   -0.010851
sensor_20_diff1   -0.002609
sensor_21_diff1   -0.001509
sensor_10_diff1    0.000000
sensor_16_diff1    0.000000
sensor_5_diff1     0.000000
sensor_1_diff1     0.000000
sensor_18_diff1    0.000000
sensor_19_diff1    0.000000
sensor_6_diff1     0.000002
sensor_15_diff1    0.000510
sensor_8_diff1     0.000923
sensor_13_diff1    0.000963
sensor_11_diff1    0.004077
sensor_2_diff1     0.006452
sensor_17_diff1    0.021918
sensor_3_diff1     0.078337
sensor_4_diff1     0.138499
sensor_14_diff1    0.149513
sensor_9_diff1     0.211973
dtype: float64

In [29]:
comparison = train_df[
    [
        "unit",
        "cycle",
        "sensor_2",
        "sensor_2_lag1",
        "sensor_2_diff1"
    ]
].head(20)

comparison

,unit,cycle,sensor_2,sensor_2_lag1,sensor_2_diff1
0,1,1,641.82,NaN,NaN
1,1,2,642.15,641.82,0.33
2,1,3,642.35,642.15,0.20
3,1,4,642.35,642.35,0.00
4,1,5,642.37,642.35,0.02
5,1,6,642.10,642.37,-0.27
6,1,7,642.48,642.10,0.38
7,1,8,642.56,642.48,0.08
8,1,9,642.12,642.56,-0.44
9,1,10,641.71,642.12,-0.41


In [30]:
print("Difference features created:", len(diff_cols))
print("Difference features:", diff_cols)

print(
    "Expected NaNs per difference feature:",
    train_df[diff_cols].isna().sum().unique()
)

Difference features created: 21
Difference features: ['sensor_1_diff1', 'sensor_2_diff1', 'sensor_3_diff1', 'sensor_4_diff1', 'sensor_5_diff1', 'sensor_6_diff1', 'sensor_7_diff1', 'sensor_8_diff1', 'sensor_9_diff1', 'sensor_10_diff1', 'sensor_11_diff1', 'sensor_12_diff1', 'sensor_13_diff1', 'sensor_14_diff1', 'sensor_15_diff1', 'sensor_16_diff1', 'sensor_17_diff1', 'sensor_18_diff1', 'sensor_19_diff1', 'sensor_20_diff1', 'sensor_21_diff1']
Expected NaNs per difference feature: [100]
